# 🧠 Meningioma Modelling Notebook

Run from the **repo root** (`meningioma-atypier/`). Consumes `output/datasets/` from the cleaning notebook.

EDA on **unimputed** data. Multivariable modelling on **imputed** data. DDA lives in the cleaning notebook.

<details>
<summary><b>Pipeline map</b> — notebook step → module</summary>

| Step | What it does | Driven by |
|------|--------------|-----------|
| 00 | Setup — imports, keep `output/` | `config/` loader + phase modules |
| 01 | Load handoff (data + schema) | `dataset_handoff.load_modelling_handoff` |
| 02 | EDA / model variant lists | `config/analysis.py` |
| 03 | EDA + diagnostic accuracy | `eda` · `diagnostic_accuracy` |
| 04 | Multivariable logistic (Rubin pool) | `inferential` → `output/inferential/` (incl. `model_artifacts/`) |
| 05 | HTML report | `config/report_settings.py` · `report` |

Config modules live in `config/` (loaded via `load("name")`).

</details>


## 00. Setup

⚙️ Loads modelling modules. Does **not** wipe `output/` — reads cleaning handoff artifacts.

<details>
<summary>🔧 How it works</summary>

- 📦 `from heavy_machinery.config import load` plus `heavy_machinery.cleaning_phase.*` and `heavy_machinery.modelling_phase.*` imports.
- 📁 `OUTPUT_ROOT = Path("output")` — same tree as the cleaning notebook.

</details>


In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

from pathlib import Path

from IPython.display import display

from heavy_machinery.config import load
from heavy_machinery.cleaning_phase.dataset_handoff import load_modelling_handoff
from heavy_machinery.cleaning_phase.missingness_resolution import load_modeling_frames
from heavy_machinery.cleaning_phase.validation import validate_unimputed_handoff, validate_imputed_frames
from heavy_machinery.modelling_phase.eda import screen_associations
from heavy_machinery.modelling_phase.diagnostic_accuracy import screen_diagnostic_accuracy
from heavy_machinery.modelling_phase.inferential import run_inferential_stage
from heavy_machinery.modelling_phase.marker_panel import run_marker_panel

OUTPUT_ROOT = Path("output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)  # do not wipe — reads cleaning outputs

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None

### 🖼️ Figure output — this is where the 1200-dpi TIFFs come from

Every figure in the pipeline is written by `modelling_phase/plot_style.py · save_figure()`, and the cell below is the only switch that decides which files it writes.

| Profile | Files written | Cost |
|---|---|---|
| `report` *(default)* | PNG only, 200 dpi | fast; `report.html` ≈ 3 MB |
| `submission` | PNG + **1200-dpi TIF** | slow; `report.html` ≈ 68 MB |

**Why the default changed.** `report.html` embeds its figures as base64 data URIs and displays them at 36 rem (576 px). A 1200-dpi TIF *and* a 1200-dpi PNG were being rasterised and compressed for every figure on every run — roughly half the pipeline's total runtime, for pixels the browser throws away.

**Set it to `submission` when you are preparing the manuscript.** That profile writes exactly what the pipeline produced before this switch existed: the AJNR 1200-dpi line-art TIFs, unchanged. The drawing code is identical in both profiles — only the export format and dpi differ.

The same cell exists in the cleaning notebook; each notebook is a separate kernel, so both need it.


In [2]:
#🟧🟧🟧 Figure output profile — "report" (fast) or "submission" (adds 1200-dpi TIFs)
import os

FIGURE_OUTPUT = "report"
#FIGURE_OUTPUT = "submission"   # ← manuscript run: also writes the AJNR 1200-dpi TIFs

# setdefault, so `ATYPIER_FIGURES=submission jupyter nbconvert …` still wins.
os.environ.setdefault("ATYPIER_FIGURES", FIGURE_OUTPUT)

from heavy_machinery.modelling_phase.plot_style import figure_formats, figure_profile
print(f"🖼️  Figure profile: {figure_profile()} → writes {', '.join(figure_formats())}")

🖼️  Figure profile: report → writes png


## 01. Load handoff

Requires `output/datasets/` parquets and `output/schema/schema_summary.csv` from cleaning §16.

`load_modelling_handoff` loads the **unimputed** cohort (`df`) for EDA, plus the ColSpec schema and which imputation method was run.

Next cell loads `output/cleaning/schema_validation.json` and validates the unimputed handoff only. Imputed draws are validated again in §04, immediately before multivariable modelling.


In [3]:
df, schema, IMPUTATION_METHOD = load_modelling_handoff(OUTPUT_ROOT)
#df.head(3)

Loaded unimputed cohort: 352 rows, 52 schema columns
Imputation method: MICE — inferential stage (§04) pools multiple imputed draws.


In [4]:
schema_validation = validate_unimputed_handoff(df, OUTPUT_ROOT)

✅ Pandera validated unimputed handoff (EDA cohort)


In [5]:
# 🟧🟧🟧 Copy-pasteable column names from the loaded cohort
#load("analysis").print_copy_pasteable_columns(df)

### 🎯 EDA


In [6]:
EDA_TARGETS = [
    'high_grade',
    #'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis'
    ]
# Binary targets only: {column: value coded as positive (1)}.
# Omit → auto-detect: True, else 1, else last sorted level (string binaries).
EDA_POSITIVE_CLASS = {
    #'progesterone_pos': True,
    #'brain_invasion': True,
    #'hist_necrosis': True,
}
EDA_PREDICTORS = [
    #🟧🟧🟧 Demographics
    #'entry_year',
    'age',
    'tumor_episode',
    
    'sex',
    'tumor_location',
    'side',
    
    
    #🟧🟧🟧 Histological features
    #'who_grade',
    #'progesterone_pos',
    #'brain_invasion',
    #'hist_necrosis',
    
    #🟧🟧🟧 Imaging features
    'tumor_margin',
    'sinus_invasion',
    'dural_tail',
    'mass_effect',
    'calcification',
    'cystic_component',
    'mri_necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    
    'capsular_enhancement',
    'heterogeneous_enhancement',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',
    'transfalcine_extension',
    
    #🟧🟧🟧 Measurement features
    #'meningioma_count',
    'perifocal_edema',
    'edema_volume_cm3',
    'max_diameter_cm',
    'tumor_volume',
    'adc_value',
    
    #🟧🟧🟧 Derived features
    'male',
    'irregular_tumor_margin',
    'skull_base_location',
    'midline_positioning',
    'multiple_meningiomas',
    'edema_index',
    'transsinus_extension',
    'recurrent_meningioma',
    #'high_grade',
    #'ki67_mid',
    #'ki67_group',
    #'edema_index_ge1', 'edema_volume_ge3.64',  'tumor_volume_ge13.95', 'max_diameter_cm_gt6', 'max_diameter_cm_gt3'
    'edema_index_ge0.0617',
    'adc_value_le0.72'
    #'edema_volume_ge4.76',
    #'tumor_volume_ge15.1',
    #'max_diameter_cm_ge3.81',
    ]

# Redundant variants of predictors already in the sweep: Youden-derived
# dichotomisations of continuous variables. They render as exploratory
# (uncorrected) and stay out of the BH multiplicity family (spec 5.2).
# Every name here is also eda_in_derived=True, so each gets a Derived-family q.
EDA_REDUNDANT_VARIANTS = [
    'tumor_volume_ge15.1', 'adc_value_le0.72', 'max_diameter_cm_ge3.81',
    'edema_volume_ge4.76', 'edema_index_ge0.0617',
    #'male', 'irregular_tumor_margin', 'skull_base_location', 'midline_positioning',
    #'transsinus_extension', 
    #'multiple_meningiomas',
]
EDA_FDR_FAMILY = [c for c in EDA_PREDICTORS if c not in EDA_REDUNDANT_VARIANTS]

### 📚 Literature-based multivariable models

In [7]:
# 📚 Literature-based multivariable models — published predictor sets.
# Each variant gets its own EPV bar, forest plot, VIF table, and interpretation.
# Format: (id, title, link, target, [predictors])
LITERATURE_MODEL_VARIANTS = [
    # Radeesri K, Lekhavat V. Asian Pacific J Cancer Prev 2023;24(3):819-825.
    ("radeesri_2023",
     "Radeesri & Lekhavat 2023 | necrosis / hyperostosis / edema MRI model",
     "https://journal.waocp.org/article_90552.html",
     "high_grade",
     ["necrosis_or_hemorrhage", "hyperostosis", "perifocal_edema"]),

    # Spille DC, Adeli A, Sporns PB et al. Neurosurg Rev 2021;44(2):1109-1117.
    ("spille_2020",
     "Spille et al. 2020 | edema volume + enhancement pattern",
     "https://pubmed.ncbi.nlm.nih.gov/32306190/",
     "high_grade",
     ["edema_volume_cm3", "heterogeneous_enhancement"]),

    # Zhang S, Chiang GC, Knapp JM et al. J Neuroradiol 2020;47(4):272-277.
    # Morphological arm only — the SWI/QSM/ADC quantitative model is not refit.
    ("zhang_2020",
     "Zhang et al. 2020 | morphological MRI model",
     "https://pubmed.ncbi.nlm.nih.gov/31541639/",
     "high_grade",
     ["calcification", "perifocal_edema", "irregular_tumor_margin",
      "skull_base_location"]),

    # Funari A, De la Garza Ramos R, Cezayirli P et al. Neuroradiology 2023;65(3):453-462.
    # tumor_volume stays continuous — their 36.0 cc cut-point is not imported.
    ("funari_2023",
     "Funari et al. 2023 | imaging score components",
     "https://pubmed.ncbi.nlm.nih.gov/36242642/",
     "high_grade",
     ["tumor_volume", "irregular_tumor_margin", "perifocal_edema"]),

    # Kawahara Y, Nakada M, Hayashi Y et al. J Neurooncol 2012;108(1):147-152.
    ("kawahara_2012",
     "Kawahara et al. 2012 | interface + enhancement heterogeneity",
     "https://pubmed.ncbi.nlm.nih.gov/22392126/",
     "high_grade",
     ["irregular_tumor_margin", "heterogeneous_enhancement"]),

    # Lin BJ, Chou KN, Kao HW et al. J Neurosurg 2014;121(5):1201-1208.
    ("lin_2014",
     "Lin et al. 2014 | MRI grading scale components",
     "https://pubmed.ncbi.nlm.nih.gov/25148007/",
     "high_grade",
     ["age_ge75", "irregular_tumor_margin", "capsular_enhancement",
      "heterogeneous_enhancement"]),

    # Peng S, Cheng Z, Guo Z. Transl Cancer Res 2021;10(9):4057-4064.
    ("peng_2021",
     "Peng, Cheng & Guo 2021 | nomogram predictors",
     "https://tcr.amegroups.org/article/view/55552/html",
     "high_grade",
     ["irregular_tumor_margin", "cortical_destruction", "skull_base_location"]),

]

### 🧪 Experimental multivariable models


In [8]:
# 🧪 Experimental multivariable models — your own predictor sets (independent of EDA_PREDICTORS).
# Add as many as you need. Each row is one model: (id, title, link, target, [predictors]).
# Grouping in the report follows this list, not the model id string.

EXPERIMENTAL_MODEL_VARIANTS = [
    (
        "experimental_model_1",
        "model 1 | high grade",
        "",
        "high_grade",
        [
            'cystic_component',
            'cortical_destruction',
            'dural_tail',
            'tumor_volume',
            'edema_volume_cm3',
            'hyperostosis',
            'mass_effect',
            'adc_value',
            'irregular_tumor_margin',
        ],
    ),
    (
        "experimental_model_2",
        "model 2 | high grade",
        "",
        "high_grade",
        [
            'dwi_hyperintensity',
            'male',
            'heterogeneous_enhancement',
            'hemorrhage',
            'transsinus_extension',
            'age',
            'calcification',
            't2_hyperintensity',
            't1_hypointensity',
            'transfalcine_extension',

        ],
    ),
]


In [9]:
_analysis = load("analysis")

EDA_TARGETS, EDA_PREDICTORS = _analysis.resolve_eda(
    df, EDA_TARGETS, EDA_PREDICTORS, output_root=OUTPUT_ROOT,
)
EDA_FDR_FAMILY = [c for c in EDA_PREDICTORS if c not in EDA_REDUNDANT_VARIANTS]

INFERENTIAL_MODEL_VARIANTS = _analysis.resolve_inferential_variants(
    df,
    LITERATURE_MODEL_VARIANTS,
    EXPERIMENTAL_MODEL_VARIANTS,
    output_root=OUTPUT_ROOT,
)

# 🔎 top_6_variables / top_1_variable — computed, never a frozen list.
#
# Ranked by discrimination, max(AUC, 1-AUC), so a protective variable is not
# thrown away for scoring below 0.5: ADC is 0.370 here, which is 0.630 the
# other way and the second-strongest variable available.
#
# Two guards, in order: skip a derived cut-point when its continuous parent is
# a candidate, and skip anything correlated above rho 0.8 with something
# already picked, taking the next candidate that clears. Without the second,
# the best six are four tumour-size measurements at rho up to 0.91.
#
# ⚠️ These two are chosen from the same 352 patients they are then fitted on.
# The bootstrap re-runs this selection inside every resample so the optimism
# correction covers the choosing, but they are still not comparable to the
# literature models, whose predictors were fixed by other people years ago.
#
# Selection needs one complete numeric vector per candidate to rank an AUC —
# it runs on the first MICE-imputed draw, not the unimputed EDA cohort `df`.
# 14 of these 33 EDA_PREDICTORS candidates carry missing values pre-imputation,
# including tumor_volume (23 missing) and adc_value (43 missing), both of
# which win, so ranking against `df` directly raises inside roc_auc_score.
#
# The pool is EDA_PREDICTORS AFTER resolve_eda above, not the 39-column cell
# literal further up this notebook: resolve_eda already dropped 6 hidden-
# parent categorical columns (sex, tumor_location, side, tumor_episode,
# tumor_margin, sinus_invasion), so the audit trail below has 12 rows — 6
# kept, 6 dropped for cut-point/collinearity reasons — and no "not numeric"
# rows, because those 6 categoricals were never candidates by this point.
from heavy_machinery.modelling_phase import variable_selection as _vs

_selection_frame = load_modeling_frames(OUTPUT_ROOT)[0]
_y = _selection_frame[EDA_TARGETS[0]].astype("boolean").fillna(False).astype(int).to_numpy()
TOP_VARIABLES, TOP_SELECTION_AUDIT = _vs.select_variables(
    _selection_frame, _y, EDA_PREDICTORS, k=6, rho_max=0.8,
    cutpoint_parent=load("analysis").CUTPOINT_PARENT,
)
# TOP_SELECTION_AUDIT (this cell's own full-cohort walk, not the CSV
# run_comparison_stage writes later) is what lets assert_reference also check
# that the declared reference's discrimination hasn't drifted from
# analysis.REFERENCE_VARIABLE_DISCRIMINATION — not just that it is still
# ranked first.
_vs.assert_reference(TOP_VARIABLES, TOP_SELECTION_AUDIT)

INFERENTIAL_MODEL_VARIANTS = list(INFERENTIAL_MODEL_VARIANTS) + [
    _analysis.InferentialModelVariant(
        model_id="top_1_variable",
        title="Top 1 variable by discrimination | high grade",
        link="", target=EDA_TARGETS[0],
        predictors=(TOP_VARIABLES[0],), experimental=True,
    ),
    _analysis.InferentialModelVariant(
        model_id="top_6_variables",
        title="Top 6 variables by discrimination | high grade",
        link="", target=EDA_TARGETS[0],
        predictors=tuple(TOP_VARIABLES), experimental=True,
    ),
]
print(f"🔎 top 6: {TOP_VARIABLES}")

INFERENTIAL_TARGETS = _analysis.resolve_inferential_targets(df, INFERENTIAL_MODEL_VARIANTS)
# Binary inferential targets only: {column: value coded as positive (1)}. Omit → auto-detect.
INFERENTIAL_POSITIVE_CLASS = {}

🔎 top 6: ['tumor_volume', 'adc_value', 'edema_volume_cm3', 'irregular_tumor_margin', 'skull_base_location', 'cystic_component']


## 03. EDA on unimputed data

`eda.screen_associations` and `diagnostic_accuracy.screen_diagnostic_accuracy`.


Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§16) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [10]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    fdr_family=EDA_FDR_FAMILY,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

In [11]:
#🟧🟧🟧 Full table
#assoc

## 04. Multivariable modelling on imputed data

Load imputed dataset(s), Pandera-validate immediately before fitting, then `inferential.run_inferential_stage` (Rubin-pooled across MICE draws).

Writes per-variant tables and forest plots under `output/inferential/`, plus Streamlit JSON under `output/inferential/model_artifacts/`. Re-running clears stale variant files first.


In [12]:
imputed_frames = load_modeling_frames(OUTPUT_ROOT)
validate_imputed_frames(schema_validation, imputed_frames)

✅ Pandera validated 20 MICE imputed draws


In [13]:
full_inferential_table = run_inferential_stage(
    schema,
    imputed_frames=imputed_frames,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    output_root=OUTPUT_ROOT,
    selection_candidates=EDA_PREDICTORS,
    )
#full_inferential_table

🔀 Validating 11 models on 4 processes (18 cores, 2 held back)…


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


...

/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


/Users/andriszaguzovs/.pyenv/versions/3.12.13/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:226: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(y, x, x, np.ones_like(x),


## 04.5 · Marker panel — the two study aims

Takes the binary columns already in `df` and answers two questions on one cohort, writing everything to `output/panel/`:

1. **Which single sign argues hardest for the outcome?** Ranked by positive likelihood ratio.
2. **Does a combination beat any single sign?** Every pair scored and bootstrap-corrected for winner's curse, plus the count score, the fitted models, and MICE stability.

It creates no columns and searches for no cut-points — the threshold flags already exist by the time it runs.

| To change | Edit |
|---|---|
| which measurements get a cut-point searched | `meningioma-thresholder.ipynb` §02, `METRICS` |
| which cut-point is baked into a column | `meningioma-cleaning.ipynb`, `DERIVATIONS` |
| which binary columns this section uses | `MARKERS_TO_EXCLUDE` below |

Accuracy is computed on **observed** data, not the MICE draws: imputing a sign would report the accuracy of a finding nobody saw. The draws are used as a stability check instead.

In [14]:
# 🟧 FILL IN — predictors to keep out of the panel.
#    There is no pick-list: the panel takes every binary predictor the EDA
#    accuracy table carries for this target, and this set is the only lever.
#    Empty set() on purpose: bare {} is an empty dict, not an empty set.
MARKERS_TO_EXCLUDE: set[str] = set()
#    candidates: "sex_male", "hist_necrosis", "progesterone_pos"

# 🟧 FILL IN — the signs the count score counts. Pre-specified, never mined:
#    "how many of these are present?" is only defensible if the list was fixed
#    before anyone looked at how well each count scores.
#
#    Empty for now — count_score_panel() is a placeholder until this is filled.
#
#    The five published cut-points, as candidates:
#      "adc_value_le0.72", "max_diameter_cm_ge3.81", "tumor_volume_ge15.1",
#      "edema_volume_ge4.76", "edema_index_ge0.0617"
#
#    ⚠️ Three of those five are near-duplicates in this cohort — tumour volume
#    and max diameter correlate at rho ~= 0.92, edema volume and edema index at
#    ~= 0.91. Counting all five lets size and edema each vote twice, so the
#    curve rises partly for arithmetic reasons. A de-duplicated three (one size,
#    one edema, ADC) is the honest dose; five is the published set. Decide, and
#    say which in the figure note either way.
COUNT_SCORE_SIGNS: list[str] = []

panel_tables = run_marker_panel(
    df,
    target=EDA_TARGETS[0],
    accuracy_table=diag_acc,
    output_root=OUTPUT_ROOT,
    exclude=MARKERS_TO_EXCLUDE,
)

#display(panel_tables["02_marker_panel_reading_view"])
#display(panel_tables["09_selection_correction"])

## 05. Build report.html

`config/report_settings.py` + `report`. Assembles a self-contained HTML report from artifacts already in `output/` (DDA from cleaning, EDA, inferential). Launch the calculator separately: `streamlit run app.py` (reads `output/inferential/model_artifacts/`).



Builds `report.html` from artifacts already in `output/` (DDA from cleaning, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §03).
- **Module** — `load("report_settings")` → `config/report_settings.py` → `report`.


In [15]:
REPORT_TITLE = "Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Doc Arturs Balodis, Sigita Zālīte, Roberts Tumeļkāns, Valērija Aksjonova, Elizabete Stankeviča, Andris Zaguzovs"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [16]:
_report = load("report_settings")
_report.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)
_report.print_output_summary(OUTPUT_ROOT)

Report written: /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/output/report/report.html
Pruned 125 embedded figure(s), 4.8 MB reclaimed
  kept 2 figure(s) not embedded in this report:
    · panel/figures/count_score.png
    · panel/figures/lr_forest.png

📦 Pipeline outputs — /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/output
════════════════════════════════════════════════════════════════════════
📁 141 files · 11.0 MB total

🧹 Cleaning                 10 files · 115.3 KB  (9 csv, 1 json)
📋 Schema                    1 files ·   2.9 KB  (1 csv)
💾 Model datasets            3 files ·  90.5 KB  (1 json, 2 parquet)
📊 DDA                       6 files ·   4.3 KB  (6 csv) — tables: 6
🕳️ Missingness              56 files ·   3.3 MB  (30 csv, 4 json, 20 parquet, 2 png) — mice: 54, tables: 2
🔬 EDA                       3 files ·  11.7 KB  (3 csv) — tables: 3
🧮 Multivariable            50 files · 321.0 KB  (28 csv, 22 json) — model_artifacts: 11, tables: 39
🧾 Report      